# Forward Physics Verification

This notebook checks independent forward-model invariants: zero response,
linear source scaling, source superposition, electromagnetic reciprocity,
state-continuation equivalence, and homogeneous-medium travel time.


In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "tests"]
candidates.extend(parent / "tests" for parent in cwd.parents)
NOTEBOOK_DIR = next(
    (path for path in candidates if (path / "verification_utils.py").is_file()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("verification_utils.py was not found from the current directory.")
notebook_path = str(NOTEBOOK_DIR)
if notebook_path not in sys.path:
    sys.path.insert(0, notebook_path)

import verification_utils as vu

REPO_ROOT = vu.configure_local_import()
for module_name in tuple(sys.modules):
    if module_name == "DeepGPR" or module_name.startswith("DeepGPR."):
        del sys.modules[module_name]
import DeepGPR

LOADED_PACKAGE = vu.assert_local_deepgpr(DeepGPR, REPO_ROOT)
print(f"Repository root: {REPO_ROOT}")
print(f"DeepGPR package: {LOADED_PACKAGE}")


Repository root: /Users/llsra/Desktop/DeepGPR
DeepGPR package: /Users/llsra/Desktop/DeepGPR/src/DeepGPR/__init__.py


In [2]:
import torch

torch.manual_seed(2026)
DEVICE = torch.device("cpu")
CHECKS = []
METADATA = vu.runtime_metadata(DeepGPR, DEVICE)
dx, dt, pml = 0.02, 2.0e-11, 8
nx, ny, nt = 48, 72, 320
er = torch.full((nx, ny), 4.0)
se = torch.full_like(er, 2.0e-4)
wavelet = DeepGPR.wavelet.ricker(3.5e8, nt, dt, 3.0e-9).reshape(1, nt, 1)

def simulate(source_amplitudes, source_location, receiver_location, **kwargs):
    return DeepGPR.compute(
        device=DEVICE,
        dx=dx,
        dt=dt,
        source_amplitudes=source_amplitudes,
        source_location=source_location,
        receiver_location=receiver_location,
        er=kwargs.pop("er", er),
        se=kwargs.pop("se", se),
        pmlthick=kwargs.pop("pmlthick", pml),
        fdtd_order=kwargs.pop("fdtd_order", 2),
        mode=2,
        **kwargs,
    )

source_a = torch.tensor([[[24, 24, 0]]], dtype=torch.int32)
source_b = torch.tensor([[[24, 32, 0]]], dtype=torch.int32)
receivers = torch.tensor(
    [[[24, 40, 0], [20, 44, 0]]], dtype=torch.int32
)


In [3]:
zero_result = simulate(torch.zeros_like(wavelet), source_a, receivers)[-1]
vu.record_check(
    CHECKS,
    "zero source produces an exactly zero response",
    float(zero_result.abs().max()) == 0.0,
    response_absmax=float(zero_result.abs().max()),
)

response = simulate(wavelet, source_a, receivers)[-1]
scale = 2.5
scaled_response = simulate(scale * wavelet, source_a, receivers)[-1]
linearity_error = vu.relative_l2(scaled_response, scale * response)
vu.record_check(
    CHECKS,
    "source amplitude linearity",
    linearity_error < 2.0e-6,
    relative_l2=linearity_error,
    tolerance=2.0e-6,
)

wavelet_b = 0.65 * wavelet
response_a = response
response_b = simulate(wavelet_b, source_b, receivers)[-1]
combined_sources = torch.cat((source_a, source_b), dim=1)
combined_wavelets = torch.cat((wavelet, wavelet_b), dim=0)
combined_response = simulate(combined_wavelets, combined_sources, receivers)[-1]
superposition_error = vu.relative_l2(
    combined_response, response_a + response_b
)
vu.record_check(
    CHECKS,
    "multiple-source superposition",
    superposition_error < 3.0e-6,
    relative_l2=superposition_error,
    tolerance=3.0e-6,
)


[PASS] zero source produces an exactly zero response
{
  "response_absmax": 0.0
}
[PASS] source amplitude linearity
{
  "relative_l2": 5.54118210681249e-07,
  "tolerance": 2e-06
}
[PASS] multiple-source superposition
{
  "relative_l2": 9.23784265093398e-07,
  "tolerance": 3e-06
}


In [4]:
point_a = torch.tensor([[[20, 24, 0]]], dtype=torch.int32)
point_b = torch.tensor([[[30, 42, 0]]], dtype=torch.int32)
trace_ab = simulate(wavelet, point_a, point_b)[-1]
trace_ba = simulate(wavelet, point_b, point_a)[-1]
reciprocity_error = vu.relative_l2(trace_ab, trace_ba)
vu.record_check(
    CHECKS,
    "same-component source-receiver reciprocity",
    reciprocity_error < 2.0e-4,
    relative_l2=reciprocity_error,
    tolerance=2.0e-4,
)


[PASS] same-component source-receiver reciprocity
{
  "relative_l2": 2.3316301921174847e-07,
  "tolerance": 0.0002
}


In [5]:
split = 130
full_result = simulate(wavelet, source_a, receivers)
first_result = simulate(wavelet[:, :split], source_a, receivers)
second_result = simulate(
    wavelet[:, split:],
    source_a,
    receivers,
    E=first_result[1],
    H=first_result[2],
    PML=first_result[3],
)
continued_response = torch.cat((first_result[-1], second_result[-1]), dim=1)
continuation_error = vu.relative_l2(continued_response, full_result[-1])
vu.record_check(
    CHECKS,
    "state continuation matches a single uninterrupted run",
    continuation_error < 2.0e-6,
    relative_l2=continuation_error,
    tolerance=2.0e-6,
)


[PASS] state continuation matches a single uninterrupted run
{
  "relative_l2": 0.0,
  "tolerance": 2e-06
}


In [6]:
travel_nx, travel_ny, travel_nt = 64, 112, 700
travel_er = torch.full((travel_nx, travel_ny), 4.0)
travel_se = torch.zeros_like(travel_er)
travel_source = torch.tensor([[[32, 24, 0]]], dtype=torch.int32)
travel_receivers = torch.tensor(
    [[[32, 44, 0], [32, 64, 0]]], dtype=torch.int32
)
travel_wavelet = DeepGPR.wavelet.ricker(
    3.0e8, travel_nt, dt, 1.0 / 3.0e8
).reshape(1, travel_nt, 1)
expected_delta = vu.expected_travel_time(20 * dx, 4.0)
travel_rows = []
for order in (2, 4, 8):
    traces = simulate(
        travel_wavelet,
        travel_source,
        travel_receivers,
        er=travel_er,
        se=travel_se,
        pmlthick=10,
        fdtd_order=order,
    )[-1][0]
    peak_near = vu.peak_sample(traces[:, 0])
    peak_far = vu.peak_sample(traces[:, 1])
    measured_delta = (peak_far - peak_near) * dt
    error = abs(measured_delta - expected_delta)
    travel_rows.append(
        {
            "order": order,
            "peak_near": peak_near,
            "peak_far": peak_far,
            "measured_delta_s": measured_delta,
            "expected_delta_s": expected_delta,
            "absolute_error_s": error,
        }
    )
    vu.record_check(
        CHECKS,
        f"homogeneous travel time at order {order}",
        error < 2.5e-10,
        **travel_rows[-1],
        tolerance_s=2.5e-10,
    )


[PASS] homogeneous travel time at order 2
{
  "absolute_error_s": 3.1487238414783186e-11,
  "expected_delta_s": 2.6685127615852166e-09,
  "measured_delta_s": 2.6999999999999998e-09,
  "order": 2,
  "peak_far": 420,
  "peak_near": 285,
  "tolerance_s": 2.5e-10
}
[PASS] homogeneous travel time at order 4
{
  "absolute_error_s": 1.1487238414783255e-11,
  "expected_delta_s": 2.6685127615852166e-09,
  "measured_delta_s": 2.68e-09,
  "order": 4,
  "peak_far": 418,
  "peak_near": 284,
  "tolerance_s": 2.5e-10
}
[PASS] homogeneous travel time at order 8
{
  "absolute_error_s": 8.512761585216676e-12,
  "expected_delta_s": 2.6685127615852166e-09,
  "measured_delta_s": 2.66e-09,
  "order": 8,
  "peak_far": 418,
  "peak_near": 285,
  "tolerance_s": 2.5e-10
}


In [7]:
vu.save_report(
    "01_forward_physics",
    CHECKS,
    METADATA,
    extra={"travel_time_rows": travel_rows},
)
print(f"Completed {len(CHECKS)} required checks.")


Report written to /Users/llsra/Desktop/DeepGPR/tests/results/01_forward_physics.json
Completed 8 required checks.
